# Accessing [ChEMBL](https://www.ebi.ac.uk/chembl/) with BioServices

**ChEMBL** is a manually curated database of bioactive molecules with drug-like
properties maintained by the EMBL-EBI.  It contains binding, functional and
ADMET information for >2 million compounds.

This notebook shows how to:

- Retrieve molecule information and properties
- Search for molecules and targets
- Explore bioactivity data
- Visualise drug-like properties (Lipinski rule-of-five)

> **ChEMBL REST API**: https://chembl.gitbook.io/chembl-interface-documentation/web-services/chembl-data-web-services


In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
from bioservices import ChEMBL

c = ChEMBL()


## 1. Retrieving a single molecule

Every molecule in ChEMBL has a unique identifier of the form `CHEMBLnnn`.
Use `get_molecule` to fetch full information for one or more molecules.


In [ ]:
# Retrieve Aspirin (CHEMBL25)
mol = c.get_molecule("CHEMBL25")
props = mol.get("molecule_properties") or {}
print("Preferred name :", mol.get("pref_name"))
print("CHEMBL ID      :", mol.get("molecule_chembl_id"))
print("Mol. formula   :", mol.get("molecule_structures", {}).get("standard_inchi_key", "n/a"))
print("Mol. weight    :", props.get("mw_freebase"))
print("AlogP          :", props.get("alogp"))
print("H-bond donors  :", props.get("hbd"))
print("H-bond accept. :", props.get("hba"))
print("Max phase      :", mol.get("max_phase"))


In [ ]:
# Retrieve multiple molecules at once
ids = ["CHEMBL25", "CHEMBL192", "CHEMBL1642"]  # Aspirin, Sildenafil, Ibuprofen
mols = c.get_molecule(ids)
for m in mols:
    props = (m.get("molecule_properties") or {})
    print(f"{m['pref_name']:20s}  MW={props.get('mw_freebase','?'):>8}  AlogP={props.get('alogp','?'):>5}")


## 2. Searching for molecules

Use `search_molecule` to find molecules by name, synonym or partial match.


In [ ]:
# Search for molecules whose preferred name contains 'ibuprofen'
results = c.search_molecule("ibuprofen")
print(f"Found {len(results)} result(s)")
for r in results[:3]:
    print(f"  {r.get('molecule_chembl_id')}: {r.get('pref_name')}")


## 3. Approved drugs and Lipinski rule-of-five

The **Lipinski rule-of-five** predicts oral bioavailability:
MW ≤ 500 Da, AlogP ≤ 5, H-bond donors ≤ 5, H-bond acceptors ≤ 10.

Let us fetch 500 approved drugs and visualise their physico-chemical properties.


In [ ]:
drugs = c.get_approved_drugs(limit=500)
rows = []
for d in drugs:
    if not d:
        continue
    props = d.get("molecule_properties") or {}
    rows.append({
        "name": d.get("pref_name", ""),
        "chembl_id": d.get("molecule_chembl_id", ""),
        "max_phase": d.get("max_phase", 0),
        "mw": props.get("mw_freebase"),
        "alogp": props.get("alogp"),
        "hbd": props.get("hbd"),
        "hba": props.get("hba"),
        "psa": props.get("psa"),
    })
df = pd.DataFrame(rows)
df[["mw", "alogp", "hbd", "hba", "psa"]] = df[["mw", "alogp", "hbd", "hba", "psa"]].apply(pd.to_numeric, errors="coerce")
print(f"Retrieved {len(df)} approved drugs")
df[["name", "chembl_id", "max_phase", "mw", "alogp"]].head()


In [ ]:
# Scatter plot: MW vs AlogP with Lipinski thresholds
df_plot = df.dropna(subset=["mw", "alogp"])

fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(df_plot["mw"], df_plot["alogp"],
           alpha=0.35, s=20, color="steelblue", label="Approved drugs")
ax.axhline(5, color="tomato", linestyle="--", linewidth=1.2, label="AlogP = 5")
ax.axvline(500, color="orange", linestyle="--", linewidth=1.2, label="MW = 500 Da")
ax.set_xlabel("Molecular Weight (Da)", fontsize=11)
ax.set_ylabel("AlogP", fontsize=11)
ax.set_title("Approved drugs: MW vs AlogP\n(Lipinski rule-of-five thresholds)", fontsize=12)
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()


In [ ]:
# Distribution of molecular weights
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

df_plot["mw"].hist(bins=30, ax=axes[0], color="steelblue", edgecolor="white")
axes[0].axvline(500, color="tomato", linestyle="--", label="Lipinski limit")
axes[0].set_xlabel("Molecular Weight (Da)")
axes[0].set_ylabel("Count")
axes[0].set_title("Molecular Weight distribution")
axes[0].legend()

df_plot["alogp"].hist(bins=30, ax=axes[1], color="darkorange", edgecolor="white")
axes[1].axvline(5, color="tomato", linestyle="--", label="Lipinski limit")
axes[1].set_xlabel("AlogP")
axes[1].set_ylabel("Count")
axes[1].set_title("AlogP distribution")
axes[1].legend()

plt.tight_layout()
plt.show()


## 4. Searching targets

Targets in ChEMBL correspond to proteins or protein complexes against which
bioactivities have been measured.


In [ ]:
# Search for kinase targets
targets = c.search_target("kinase")
print(f"Found {len(targets)} kinase targets")
for t in targets[:5]:
    print(f"  {t.get('target_chembl_id')}: {t.get('pref_name')} ({t.get('target_type')})")


In [ ]:
# Retrieve a specific target: EGFR (CHEMBL203)
egfr = c.get_target("CHEMBL203")
print("Target name :", egfr.get("pref_name"))
print("Target type :", egfr.get("target_type"))
print("Organism    :", egfr.get("organism"))


## 5. Similarity search

Use `get_similarity` to find molecules structurally similar to a query
molecule (given as a SMILES string), above a similarity threshold.


In [ ]:
# Find molecules similar to Aspirin (SMILES: CC(=O)Oc1ccccc1C(=O)O)
# Similarity threshold: 70 %
similar = c.get_similarity("CC(=O)Oc1ccccc1C(=O)O", 70)
print(f"Found {len(similar)} similar molecules (threshold=70%)")
for s in similar[:5]:
    print(f"  {s.get('molecule_chembl_id')}: {s.get('pref_name')}  "
          f"similarity={s.get('similarity')}")


---
For more information, please see the
[bioservices.chembl module documentation](https://bioservices.readthedocs.io/en/main/references.html#module-bioservices.chembl)
and the [ChEMBL web services documentation](https://chembl.gitbook.io/chembl-interface-documentation/web-services/chembl-data-web-services).
